In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Lat-Height monthly anomalies (hindcast − multi-year climatology)
----------------------------------------------------------------
- Climatology is multi-year ONLY from MERGED_FILE.
- Hindcast:
    Feb-initial → [2,3,4,5]
    Mar-initial → [3,4,5,6]
- Processing:
    1. Take hindcast, lon-mean, member-mean → (plev, lat, time)
    2. Monthly mean → (plev, lat, month)
    3. Take longrun multi-year monthly climatology → (plev, lat, month)
    4. Align on common months and subtract: hc_mon(m) - clim_mon(m)
- Plot:
    height (pressure, log scale) × latitude, filled contours of anomaly only.
- Extra debugging prints everywhere so we can diagnose strange February behaviour.

Author: (you)
"""

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os, math, glob
from typing import List, Dict, Tuple, Union

# ----------------------------------------------------
# Global paths
# ----------------------------------------------------
MERGED_FILE = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc'

def get_year_small_pert(year: str) -> Tuple[str, str]:
    base = f'/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{year}'
    file_Feb = os.path.join(base, 'Feb', f'BWCN.e122.f19_g16.002.{year}-02.*.nc')
    file_Mar = os.path.join(base, 'Mar', f'BWCN.e122.f19_g16.002.{year}-03.*.nc')
    return file_Feb, file_Mar

def get_year_nocouple(year: str) -> Tuple[str, str]:
    file_Feb = f'/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/{year}-02/*.nc'
    file_Mar = f'/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/{year}-03/*.nc'
    return file_Feb, file_Mar

# ----------------------------------------------------
# Config
# ----------------------------------------------------
OUTPUT_DIR = '/home/weiji/restart_exam/plots/LatHeight_Profiles_mixingration/'
VARIABLES = ['O3', 'T', 'U']
N_LEVELS = 20

MONTH_NAMES = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

# ----------------------------------------------------
# Hindcast runs（你可以继续往下加别的年）
#   'files' 可以是单字符串或列表（比如 Feb init 包含 Feb, Mar, Apr, May 目录）
#   Feb init: 2–5 月；Mar init: 3–6 月
# ----------------------------------------------------
file_0008_Feb_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc'
file_0008_Mar_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc'
file_0008_Feb_nocpl = '/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc'
file_0008_Mar_nocpl = '/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc'

hindcast_runs: List[Dict] = [
    {
        'label': '0008 small pertlim (Feb init)',
        'init': 'Feb',
        'year': '0008',
        'files': [file_0008_Feb_small, file_0008_Mar_small]
        # 如果有 Apr/May 的同类目录，继续在 list 里 append
    },
    {
        'label': '0008 nocouple (Feb init)',
        'init': 'Feb',
        'year': '0008',
        'files': [file_0008_Feb_nocpl, file_0008_Mar_nocpl]
    },
    {
        'label': '0008 small pertlim (Mar init)',
        'init': 'Mar',
        'year': '0008',
        'files': [file_0008_Mar_small]
        # 同理可以加 Apr/May/Jun
    },
    {
        'label': '0008 nocouple (Mar init)',
        'init': 'Mar',
        'year': '0008',
        'files': [file_0008_Mar_nocpl]
    },
]

# ----------------------------------------------------
# Helpers
# ----------------------------------------------------
def months_for_init(init: str) -> List[int]:
    if init.lower().startswith('feb'):
        return [2,3,4,5]
    elif init.lower().startswith('mar'):
        return [3,4,5,6]
    else:
        raise ValueError("init must be 'Feb' or 'Mar'")

def auto_subplot_layout(n: int) -> Tuple[int,int]:
    if n <= 4:
        return (1, n)
    cols = int(math.ceil(math.sqrt(n)))
    rows = int(math.ceil(n / cols))
    return rows, cols

def _open_many(patterns: Union[str, List[str]]) -> xr.Dataset:
    """
    Expand all globs -> file list -> open.
    We'll first try combine='by_coords', then fallback to concat_dim='member'.
    """
    if isinstance(patterns, str):
        paths = sorted(glob.glob(patterns))
    else:
        paths = []
        for p in patterns:
            paths.extend(glob.glob(p))
        paths = sorted(paths)

    if not paths:
        raise FileNotFoundError(f"No files matched: {patterns}")

    try:
        ds = xr.open_mfdataset(paths, combine='by_coords')
    except Exception:
        ds = xr.open_mfdataset(paths, concat_dim='member', combine='nested')

    return ds

def process_var_latheight(ds: xr.Dataset, var: str) -> xr.DataArray:
    """
    Take dataset -> pick var -> zonal mean (lon) -> ensemble mean if member present.
    Return dims in order (plev, lat, time).
    """
    if var not in ds:
        raise KeyError(f"Variable {var} not found in dataset.")
    da = ds[var].mean(dim='lon', keep_attrs=True)
    if 'member' in da.dims:
        da = da.mean(dim='member', keep_attrs=True)
    wanted = [d for d in ['plev','lat','time'] if d in da.dims]
    da = da.transpose(*wanted)
    return da

# ----------------------------------------------------
# Climatology (multi-year only)
# ----------------------------------------------------
def read_longrun_climatology_monthly(var: str) -> xr.DataArray:
    """
    From MERGED_FILE:
    - Do the same zonal/member averaging
    - Group by calendar month -> multi-year mean
    Return DataArray (plev, lat, month)
    """
    ds = xr.open_dataset(MERGED_FILE)
    da = process_var_latheight(ds, var)
    ds.close()

    clim_mon = da.groupby('time.month').mean('time', keep_attrs=True)
    clim_mon = clim_mon.transpose('plev','lat','month')

    # Debug climatology info
    print(f"[CLIM DEBUG] var={var} climatology shape={clim_mon.shape} dims={clim_mon.dims}")
    months_available = clim_mon['month'].values.tolist()
    print(f"[CLIM DEBUG] var={var} climatology months_available={months_available}")
    return clim_mon

# ----------------------------------------------------
# Hindcast reading and monthly mean
# ----------------------------------------------------
def read_hindcast_monthly(var: str, files: Union[str, List[str]]) -> xr.DataArray:
    """
    Read hindcast, zonal mean & member mean -> (plev,lat,time)
    Then groupby month -> monthly mean
    Return (plev,lat,month)
    """
    ds = _open_many(files)
    da = process_var_latheight(ds, var)
    ds.close()

    # Debug raw hindcast info
    tvals = da['time'].values
    months_here = da['time'].dt.month.values
    unique_months = np.unique(months_here)
    print(f"[HIN DEBUG] var={var} raw hindcast time range: {str(tvals[0])} to {str(tvals[-1])}")
    print(f"[HIN DEBUG] var={var} raw hindcast months: {unique_months.tolist()}")

    # number of timesteps per month in hindcast
    for m in unique_months:
        count_m = np.sum(months_here == m)
        print(f"[HIN DEBUG] var={var} month {m} has {int(count_m)} timesteps in hindcast")

    hc_mon = da.groupby('time.month').mean('time', keep_attrs=True)
    hc_mon = hc_mon.transpose('plev','lat','month')

    # Debug hindcast-monthly stats before subtraction
    for m in hc_mon['month'].values.tolist():
        field = hc_mon.sel(month=m).values
        print(
            f"[HIN DEBUG] var={var} month {m} hc_mon stats: "
            f"mean={np.nanmean(field):.4g}, std={np.nanstd(field):.4g}, "
            f"min={np.nanmin(field):.4g}, max={np.nanmax(field):.4g}"
        )

    return hc_mon

# ----------------------------------------------------
# Compute anomaly and debug
# ----------------------------------------------------
def compute_anomaly_vs_clim(
    hc_mon: xr.DataArray,
    clim_mon: xr.DataArray,
    months_req: List[int],
    var: str
) -> Tuple[xr.DataArray, List[int]]:
    """
    Align on month indices and subtract climatology.

    Steps:
    1. figure out which months we *want* (months_req)
    2. figure out which months hindcast actually has (hc_have)
    3. figure out which months climatology has (clim_have)
    4. take intersection -> used
    5. anomaly = hc_mon.sel(month=used) - clim_mon.sel(month=used)

    We'll also:
    - print coordinate equality diagnostics for lat/plev
    - print stats of clim_mon and anomaly per used month
    """

    hc_have = hc_mon['month'].values.tolist()
    clim_have = clim_mon['month'].values.tolist()
    used = sorted(set(months_req).intersection(hc_have).intersection(clim_have))

    if not used:
        raise RuntimeError(
            f"[ERROR] No overlapping months. "
            f"requested={months_req}, hc_have={hc_have}, clim_have={clim_have}"
        )

    # Check coord compatibility before subtract
    # (same vertical levels and same lat grid?)
    hc_lat = hc_mon['lat'].values
    cl_lat = clim_mon['lat'].values
    hc_plev = hc_mon['plev'].values
    cl_plev = clim_mon['plev'].values

    same_lat = np.array_equal(hc_lat, cl_lat)
    same_plev = np.array_equal(hc_plev, cl_plev)

    print(f"[ALIGN DEBUG] var={var} requested={months_req}, hc_have={hc_have}, clim_have={clim_have}, used={used}")
    print(f"[ALIGN DEBUG] var={var} hc_mon shape={hc_mon.shape}, clim_mon shape={clim_mon.shape}")
    print(f"[ALIGN DEBUG] var={var} lat equal? {same_lat}, "
          f"hc_lat.shape={hc_lat.shape}, clim_lat.shape={cl_lat.shape}")
    print(f"[ALIGN DEBUG] var={var} plev equal? {same_plev}, "
          f"hc_plev.shape={hc_plev.shape}, clim_plev.shape={cl_plev.shape}")

    if (not same_lat) or (not same_plev):
        print("[WARN] lat or plev grid not identical between hindcast and climatology. "
              "Anomaly will likely blow up or broadcast strangely.")
        # We do a straight subtraction assuming alignment by coordinate labels.
        # xarray will try to align by coordinate values. If grids differ, result may be sparse.
        # We'll just rely on .sel(month=used) - .sel(month=used) which will align by coords.

    # Actually compute anomaly
    anom = hc_mon.sel(month=used) - clim_mon.sel(month=used)

    # Debug: stats of clim and anomaly
    for m in used:
        clim_field = clim_mon.sel(month=m).values
        anom_field = anom.sel(month=m).values
        print(
            f"[CLIM DEBUG] var={var} month {m} clim stats: "
            f"mean={np.nanmean(clim_field):.4g}, std={np.nanstd(clim_field):.4g}, "
            f"min={np.nanmin(clim_field):.4g}, max={np.nanmax(clim_field):.4g}"
        )
        print(
            f"[ANOM DEBUG] var={var} month {m} anomaly stats: "
            f"mean={np.nanmean(anom_field):.4g}, std={np.nanstd(anom_field):.4g}, "
            f"max|.|={np.nanmax(np.abs(anom_field)):.4g}"
        )
        if m == 2:
            print(
                f"[ANOM DEBUG FEB CHECK] var={var} Feb anomaly stats again: "
                f"mean={np.nanmean(anom_field):.4g}, std={np.nanstd(anom_field):.4g}, "
                f"max|.|={np.nanmax(np.abs(anom_field)):.4g}"
            )

    # Make sure dims = (plev, lat, month)
    anom = anom.transpose('plev','lat','month')
    return anom, used

# ----------------------------------------------------
# Plotting helper
# ----------------------------------------------------
def plot_latheight_anomaly(
    anom: xr.DataArray,
    months: List[int],
    var: str,
    run_label: str,
    init_label: str,
    year: str,
    save_path: str,
    n_levels: int = N_LEVELS
):
    """
    Draw 1 panel per month in `months`.
    Filled contour of anomaly. No climatology contour overlay.
    y-axis is pressure, log scale, inverted.
    x-axis is latitude.
    """
    data = anom.values  # shape (plev, lat, month)
    vmax = np.nanmax(np.abs(data))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    vmin = -vmax

    rows, cols = auto_subplot_layout(len(months))
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3.6*rows), squeeze=False)

    plevels = anom['plev'].values
    lats    = anom['lat'].values
    LAT, PLEV = np.meshgrid(lats, plevels)

    cf_last = None
    for i, m in enumerate(months):
        r = i // cols
        c = i % cols
        ax = axes[r][c]

        field = anom.sel(month=m).values
        cf_last = ax.contourf(
            LAT, PLEV, field,
            levels=n_levels,
            cmap='RdBu_r',
            vmin=vmin, vmax=vmax
        )

        ax.set_yscale('log')
        ax.invert_yaxis()
        ax.set_xlabel('Latitude (deg)')
        ax.set_ylabel('Pressure (Pa)')
        ax.set_xlim(lats.min(), lats.max())
        ax.set_title(f"{MONTH_NAMES.get(int(m), str(m))}", fontsize=11)

    # hide unused subplots
    for j in range(len(months), rows*cols):
        r = j // cols
        c = j % cols
        axes[r][c].axis('off')

    if cf_last is not None:
        cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        fig.colorbar(cf_last, cax=cax, label=f'{var} Anomaly (hindcast − multi-year clim)')

    fig.suptitle(
        f"{run_label} — {var} Lat-Height Anomaly\nInit={init_label}  Year={year}",
        fontsize=14
    )
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=160, bbox_inches='tight')
    plt.close(fig)
    print(f"[INFO] Saved figure: {save_path}")

# ----------------------------------------------------
# Driver
# ----------------------------------------------------
def generate_all_latheight():
    print("[INFO] Loading multi-year climatology (monthly means)...")
    clim_all = {var: read_longrun_climatology_monthly(var) for var in VARIABLES}
    print("[INFO] Done loading climatology.\n")

    for run in hindcast_runs:
        run_label   = run.get('label', 'hindcast')
        init_label  = run['init']
        year        = run.get('year', 'NA')
        files       = run['files']
        months_req  = months_for_init(init_label)

        for var in VARIABLES:
            print("="*80)
            print(f"[RUN START] {run_label} | Var={var} | Init={init_label} | Year={year}")
            print(f"[RUN INFO ] months_req={months_req}")
            print("-"*80)

            # 1. Hindcast → monthly mean
            hc_mon = read_hindcast_monthly(var, files)

            # 2. Compute anomaly vs multi-year clim
            clim_mon = clim_all[var]
            anom, months_used = compute_anomaly_vs_clim(hc_mon, clim_mon, months_req, var)

            print(f"[FINAL DEBUG] var={var} months_used={months_used}")
            print(f"[FINAL DEBUG] var={var} anomaly shape={anom.shape} dims={anom.dims}")
            print("="*80 + "\n")

            # 3. Plot
            out_name = f"{run_label.replace(' ', '_').replace('/', '-')}_{var}.png"
            save_path = os.path.join(OUTPUT_DIR, out_name)
            plot_latheight_anomaly(anom, months_used, var, run_label, init_label, year, save_path)

    print("[INFO] All lat-height anomaly figures generated.\n")

if __name__ == "__main__":
    generate_all_latheight()
